## 📊 Resumo Final e Conclusões### 🎯 Principais Resultados#### Fase 1 - Modelos Baseline:- **Melhor Modelo**: Random Forest- **Recall**: 0.91- **F1-Score**: 0.72- **Limiar**: 0.30 (fixo)#### Fase 2 - Otimização com AG:- **Modelo**: Random Forest Otimizado (AG)- **Recall**: 0.94 ✅ **(+3.3% de melhoria)**- **F1-Score**: 0.66- **Limiar**: 0.20 (otimizado pelo AG)### 🔍 Insights Principais1. **Otimização Bem-Sucedida**: O AG conseguiu melhorar o Recall de 0.91 para 0.94, detectando **94% dos casos reais de diabetes**.2. **Limiar Otimizado**: O AG descobriu que o limiar ideal é **0.20** (em vez de 0.30), priorizando a detecção de casos positivos.3. **Trade-off Recall vs F1**: O F1-score diminuiu de 0.72 para 0.66 devido ao aumento de falsos positivos. Isso é **esperado e desejável** em diagnósticos médicos, onde **não detectar um diabético é mais grave** que um falso alarme.4. **Features Mais Importantes**:   - **glucose_bmi** (Interação Glicose x IMC) - Fator de maior peso   - **Glucose** (Nível de Glicose Sanguínea)   - **idade_bmi** (Interação Idade x IMC)5. **Interpretabilidade**: A integração com LLM (Gemini) permite traduzir resultados técnicos em recomendações médicas acionáveis.### 🚀 Próximos Passos1. **Validação Clínica**: Testar o modelo em dados reais de hospitais2. **Monitoramento Contínuo**: Implementar pipeline de retreinamento periódico3. **Expansão de Features**: Incluir dados de exames laboratoriais adicionais4. **Interface Médica**: Desenvolver dashboard interativo para profissionais de saúde### 📚 Referências- Dataset: [Pima Indians Diabetes Database (Kaggle)](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database)- DEAP: [Distributed Evolutionary Algorithms in Python](https://deap.readthedocs.io/)- SHAP: [SHapley Additive exPlanations](https://shap.readthedocs.io/)- Gemini API: [Google Generative AI](https://ai.google.dev/)---**Desenvolvido por**: Grupo Tech Challenge Fase 2**Data**: 2024

In [ ]:
top_features = importancia_features.head(3).index.tolist()prompt_texto = f"""Você é um assistente médico especialista. Analise o desempenho e os fatores de risco (features) do modelo de diagnóstico de diabetes.Modelo: Random Forest Otimizado (AG)Métricas de Desempenho:- Recall (Sensibilidade): {recall_otimizado:.2f}- F1-score: {f1_otimizado:.2f}- Acurácia: {acc_otimizado:.2f}Fatores de Risco (Importância SHAP/Clássica):1. {top_features[0]}2. {top_features[1]}3. {top_features[2]}Gere um relatório estruturado em linguagem natural. A análise deve focar em:1. Um resumo da performance do modelo, com ênfase no Alto Recall.2. Insights Clínicos Acionáveis: O que o médico deve fazer para cada um dos 3 fatores de risco."""print("🧠 Gerando Relatório Interpretável via LLM...\n")print("="*80)if CHAMADA_LLM_ATIVA:    try:        response = client.models.generate_content(            model=MODELO_LLM,            contents=prompt_texto,            config=types.GenerateContentConfig(                system_instruction="Você é um assistente médico profissional, conciso e focado em gerar insights acionáveis. Sua saída deve ser formatada como um relatório clínico.",                temperature=0.2            )        )        print("\n*** Relatório de Diagnóstico Otimizado (Gerado por LLM - Gemini API) ***\n")        print(response.text)        print("\n   [✅ SUCESSO] Relatório gerado pela API Gemini.")    except Exception as e:        print(f"   [❌ FALHA API] Erro ao chamar Gemini: {e}")        CHAMADA_LLM_ATIVA = Falseif not CHAMADA_LLM_ATIVA:    print("\n*** Relatório de Diagnóstico Otimizado (SIMULAÇÃO) ***\n")    print(f"**Modelo Base:** Random Forest Otimizado (AG)")    print(f"\n**Performance (Foco em Sensibilidade - Recall):**")    print(f"- **Recall (Capacidade de detectar positivos):** {recall_otimizado:.2f}")    print(f"- **F1-score (Equilíbrio entre Precisão e Recall):** {f1_otimizado:.2f}")    print(f"- **Acurácia Geral:** {acc_otimizado:.2f}")    print(f"\n**🔬 Insights Clínicos Acionáveis:**")    print(f"\n1. **{top_features[0]}:** Fator de maior peso. Requer atenção imediata.")    print(f"2. **{top_features[1]}:** Segundo fator mais influente. Monitoramento rigoroso.")    print(f"3. **{top_features[2]}:** Modificador de risco. Rastreamento anual recomendado.")    print("\n   [🛠️ SIMULAÇÃO] Relatório gerado por template.")print("\n" + "="*80)

In [ ]:
import osfrom dotenv import load_dotenvload_dotenv()try:    from google import genai    from google.genai import types        api_key = os.getenv('GEMINI_API_KEY')    if api_key:        client = genai.Client(api_key=api_key)        MODELO_LLM = "gemini-2.5-flash-lite"        CHAMADA_LLM_ATIVA = True        print("✅ Gemini Client configurado. Chamadas à API ativas.")    else:        raise ValueError("GEMINI_API_KEY não encontrada")except Exception as e:    CHAMADA_LLM_ATIVA = False    print(f"⚠️ Alerta: Gemini API não configurada ({e}). Usando modo de SIMULAÇÃO LLM.")

## 🧠 Interpretabilidade com LLM (Gemini)Agora vamos usar um Large Language Model (Gemini) para traduzir os resultados técnicos em insights médicos acionáveis.**Nota**: Para executar esta seção, você precisa configurar a variável de ambiente `GEMINI_API_KEY` no arquivo `.env`.

In [ ]:
print("📊 SHAP Bar Plot (Importância Média Absoluta):\n")plt.figure(figsize=(10, 8))shap.summary_plot(shap_values, X_teste_df, feature_names=nomes_features, plot_type="bar", show=False)plt.title('SHAP Bar Plot - Importância Média Absoluta', fontsize=14, fontweight='bold')plt.tight_layout()plt.show()shap_importance = np.abs(shap_values).mean(axis=0)shap_importance_df = pd.DataFrame({    'Feature': nomes_features,    'SHAP Importance': shap_importance}).sort_values('SHAP Importance', ascending=False)print("\n📊 Top 5 Features (SHAP):\n")print(shap_importance_df.head())

In [ ]:
print("📊 SHAP Summary Plot (Importância e Impacto Global):\n")plt.figure(figsize=(10, 8))shap.summary_plot(shap_values, X_teste_df, feature_names=nomes_features, show=False)plt.title('SHAP Summary Plot - Impacto Global das Features', fontsize=14, fontweight='bold')plt.tight_layout()plt.show()print("\n🔍 Interpretação:")print("   - Pontos vermelhos (alto valor da feature) à direita aumentam a chance de diabetes")print("   - Pontos azuis (baixo valor) à esquerda diminuem a chance de diabetes")print("   - Quanto mais disperso verticalmente, maior a variabilidade do impacto")

In [ ]:
X_teste_df = pd.DataFrame(X_teste, columns=nomes_features)print("🔍 Calculando valores SHAP...")explainer = shap.TreeExplainer(modelo_otimizado, check_additivity=False)shap_values = explainer.shap_values(X_teste_df)if isinstance(shap_values, list):    shap_values = shap_values[1]print("✅ Valores SHAP calculados!")importancias = modelo_otimizado.feature_importances_importancia_features = pd.Series(importancias, index=nomes_features).sort_values(ascending=False)print("\n📊 Top 5 Features (Importância Clássica):\n")print(importancia_features.head())plt.figure(figsize=(10, 6))sns.barplot(x=importancia_features.head(10).values, y=importancia_features.head(10).index, palette="viridis")plt.title('Top 10 Features - Importância Clássica (Random Forest)', fontsize=14, fontweight='bold')plt.xlabel('Importância')plt.tight_layout()plt.show()

## 🔍 Interpretabilidade: Análise SHAPSHAP (SHapley Additive exPlanations) nos ajuda a entender quais features são mais importantes para as predições do modelo.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))cm_baseline = confusion_matrix(y_teste, resultados_baseline['Random Forest']['y_pred'])sns.heatmap(cm_baseline, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=False)axes[0].set_title(f'Random Forest Baseline\nRecall: {resultados_baseline["Random Forest"]["Recall"]:.2f} | Limiar: {limiar_baseline}',                   fontsize=12, fontweight='bold')axes[0].set_xlabel('Predito')axes[0].set_ylabel('Real')cm_otimizado = confusion_matrix(y_teste, y_pred_otimizado)sns.heatmap(cm_otimizado, annot=True, fmt='d', cmap='Greens', ax=axes[1], cbar=False)axes[1].set_title(f'Random Forest Otimizado (AG)\nRecall: {recall_otimizado:.2f} | Limiar: {limiar_otimizado:.2f}',                   fontsize=12, fontweight='bold')axes[1].set_xlabel('Predito')axes[1].set_ylabel('Real')plt.suptitle('Comparação: Matrizes de Confusão', fontsize=16, fontweight='bold', y=1.02)plt.tight_layout()plt.show()print("📊 Análise das Matrizes de Confusão:")print(f"\n   Baseline:")print(f"      - Verdadeiros Positivos (TP): {cm_baseline[1,1]}")print(f"      - Falsos Negativos (FN): {cm_baseline[1,0]}")print(f"      - Falsos Positivos (FP): {cm_baseline[0,1]}")print(f"\n   Otimizado (AG):")print(f"      - Verdadeiros Positivos (TP): {cm_otimizado[1,1]}")print(f"      - Falsos Negativos (FN): {cm_otimizado[1,0]}")print(f"      - Falsos Positivos (FP): {cm_otimizado[0,1]}")print(f"\n   ✅ O modelo otimizado detectou {cm_otimizado[1,1] - cm_baseline[1,1]} casos adicionais de diabetes!")

In [ ]:
y_proba_otimizado = modelo_otimizado.predict_proba(X_teste)[:, 1]y_pred_otimizado = (y_proba_otimizado >= limiar_otimizado).astype(int)recall_otimizado = recall_score(y_teste, y_pred_otimizado)f1_otimizado = f1_score(y_teste, y_pred_otimizado)acc_otimizado = accuracy_score(y_teste, y_pred_otimizado)print("📊 Resultados do Modelo Otimizado (AG):\n")print(f"   Recall (Sensibilidade): {recall_otimizado:.2f}")print(f"   F1-Score: {f1_otimizado:.2f}")print(f"   Acurácia: {acc_otimizado:.2f}")print(f"   Limiar: {limiar_otimizado:.2f}")print("\n" + "="*80)print("\n📊 Comparação: Random Forest Baseline vs Otimizado (AG)\n")comparacao = pd.DataFrame({    'Random Forest (Baseline)': [        resultados_baseline['Random Forest']['Recall'],        resultados_baseline['Random Forest']['F1-Score'],        resultados_baseline['Random Forest']['Acurácia'],        limiar_baseline    ],    'Random Forest Otimizado (AG)': [        recall_otimizado,        f1_otimizado,        acc_otimizado,        limiar_otimizado    ]}, index=['Recall', 'F1-Score', 'Acurácia', 'Limiar'])comparacao['Diferença'] = comparacao['Random Forest Otimizado (AG)'] - comparacao['Random Forest (Baseline)']comparacao['Melhoria (%)'] = (comparacao['Diferença'] / comparacao['Random Forest (Baseline)']) * 100print(comparacao)print("\n" + "="*80)print(f"\n🏆 Melhoria no Recall: {comparacao.loc['Recall', 'Diferença']:.4f} (+{comparacao.loc['Recall', 'Melhoria (%)']:.1f}%)")print(f"🎯 Limiar otimizado: {limiar_baseline:.2f} → {limiar_otimizado:.2f}")

## 📊 Comparação: Baseline vs Otimizado (AG)Vamos comparar o desempenho do modelo baseline com o modelo otimizado pelo AG.

In [ ]:
gen_data = log.select("gen", "avg", "max", "min")fig, ax = plt.subplots(figsize=(12, 6))generations = [entry[0] for entry in gen_data]avg_fitness = [entry[1] for entry in gen_data]max_fitness = [entry[2] for entry in gen_data]min_fitness = [entry[3] for entry in gen_data]ax.plot(generations, avg_fitness, label='Média', linewidth=2, marker='o')ax.plot(generations, max_fitness, label='Máximo', linewidth=2, marker='s', linestyle='--')ax.plot(generations, min_fitness, label='Mínimo', linewidth=2, marker='^', linestyle=':')ax.set_xlabel('Geração', fontsize=12)ax.set_ylabel('Fitness (Recall)', fontsize=12)ax.set_title('Evolução do Fitness ao Longo das Gerações', fontsize=14, fontweight='bold')ax.legend(fontsize=10)ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()print(f"📊 Evolução do AG:")print(f"   - Recall inicial (Geração 0): {gen_data[0][1]:.4f}")print(f"   - Recall final (Geração {geracoes}): {gen_data[-1][2]:.4f}")print(f"   - Melhoria: {(gen_data[-1][2] - gen_data[0][1]):.4f} (+{((gen_data[-1][2] - gen_data[0][1])/gen_data[0][1]*100):.1f}%)")

In [ ]:
melhor_individuo = hof[0]params_otimizados = {    'n_estimators': int(melhor_individuo[0]),    'max_depth': int(melhor_individuo[1]),    'min_samples_split': int(melhor_individuo[2]),    'min_samples_leaf': int(melhor_individuo[3]),    'max_features': melhor_individuo[4],    'bootstrap': bool(melhor_individuo[5]),    'class_weight': 'balanced',    'random_state': 42,    'n_jobs': -1}limiar_otimizado = max(0.2, min(0.5, melhor_individuo[6]))print("🏆 Melhor Indivíduo Encontrado:\n")print(f"   ✅ Recall (CV): {melhor_individuo.fitness.values[0]:.4f}")print(f"   🎯 Limiar Otimizado: {limiar_otimizado:.4f}")print(f"\n   🧬 Hiperparâmetros Otimizados:")for param, valor in params_otimizados.items():    if param not in ['class_weight', 'random_state', 'n_jobs']:        if isinstance(valor, float):            print(f"      - {param}: {valor:.4f}")        else:            print(f"      - {param}: {valor}")modelo_otimizado = RandomForestClassifier(**params_otimizados)modelo_otimizado.fit(X_treino_res, y_treino_res)print(f"\n✅ Modelo Random Forest Otimizado treinado com sucesso!")

In [ ]:
np.random.seed(42)geracoes = 30populacao = 50print(f"🧬 Iniciando Evolução do Algoritmo Genético...")print(f"   - População: {populacao} indivíduos")print(f"   - Gerações: {geracoes}")print(f"   - Objetivo: Maximizar Recall\n")pop = toolbox.population(n=populacao)hof = tools.HallOfFame(1)stats = tools.Statistics(lambda ind: ind.fitness.values)stats.register("avg", np.mean)stats.register("max", np.max)stats.register("min", np.min)print("🔬 Evolução em andamento...\n")pop, log = algorithms.eaSimple(pop, toolbox, cxpb=0.7, mutpb=0.2,                                 ngen=geracoes, stats=stats,                                 halloffame=hof, verbose=True)print("\n✅ Evolução concluída!")

In [ ]:
def avaliar_individuo(individual):    """    Função fitness: Avalia um conjunto de hiperparâmetros + limiar.    Retorna o Recall médio via validação cruzada (3-fold).    """    n_estimators = max(50, min(500, int(individual[0])))    max_depth = max(5, min(30, int(individual[1])))    min_samples_split = max(2, min(20, int(individual[2])))    min_samples_leaf = max(1, min(10, int(individual[3])))    max_features = max(0.3, min(1.0, individual[4]))    bootstrap = bool(individual[5])    threshold = max(0.2, min(0.5, individual[6]))        try:        modelo = RandomForestClassifier(            n_estimators=n_estimators,            max_depth=max_depth,            min_samples_split=min_samples_split,            min_samples_leaf=min_samples_leaf,            max_features=max_features,            bootstrap=bootstrap,            class_weight='balanced',            random_state=42,            n_jobs=-1        )                skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)        recalls = []                for train_idx, val_idx in skf.split(X_treino_res, y_treino_res):            X_train_fold = X_treino_res[train_idx]            y_train_fold = y_treino_res[train_idx]            X_val_fold = X_treino_res[val_idx]            y_val_fold = y_treino_res[val_idx]                        modelo.fit(X_train_fold, y_train_fold)            y_proba = modelo.predict_proba(X_val_fold)[:, 1]            y_pred = (y_proba >= threshold).astype(int)            recall = recall_score(y_val_fold, y_pred)            recalls.append(recall)                recall_medio = np.mean(recalls)        return (recall_medio,)    except:        return (0.0,)toolbox.register("evaluate", avaliar_individuo)toolbox.register("mate", tools.cxTwoPoint)def mutacao_customizada(individual, indpb):    """Mutação com limites para evitar valores inválidos"""    for i in range(len(individual)):        if np.random.random() < indpb:            if i == 0:                individual[i] += np.random.normal(0, 50)            elif i == 1:                individual[i] += np.random.normal(0, 3)            elif i == 2:                individual[i] += np.random.normal(0, 2)            elif i == 3:                individual[i] += np.random.normal(0, 1)            elif i == 4:                individual[i] += np.random.normal(0, 0.1)            elif i == 5:                individual[i] = np.random.choice([True, False])            elif i == 6:                individual[i] += np.random.normal(0, 0.05)    return (individual,)toolbox.register("mutate", mutacao_customizada, indpb=0.3)toolbox.register("select", tools.selTournament, tournsize=3)print("✅ Operadores genéticos registrados!")print("   - Crossover: Two-Point (70%)")print("   - Mutação: Customizada (20%, indpb=0.3)")print("   - Seleção: Torneio (tournsize=3)")

In [ ]:
from deap import base, creator, tools, algorithmsfrom sklearn.model_selection import StratifiedKFoldprint("🧬 Configurando Algoritmo Genético...\n")if hasattr(creator, "FitnessMax"):    del creator.FitnessMaxif hasattr(creator, "Individual"):    del creator.Individualcreator.create("FitnessMax", base.Fitness, weights=(1.0,))creator.create("Individual", list, fitness=creator.FitnessMax)toolbox = base.Toolbox()toolbox.register("attr_n_estimators", np.random.randint, 100, 500)toolbox.register("attr_max_depth", np.random.randint, 5, 30)toolbox.register("attr_min_samples_split", np.random.randint, 2, 20)toolbox.register("attr_min_samples_leaf", np.random.randint, 1, 10)toolbox.register("attr_max_features", np.random.uniform, 0.3, 1.0)toolbox.register("attr_bootstrap", np.random.choice, [True, False])toolbox.register("attr_threshold", np.random.uniform, 0.2, 0.5)toolbox.register("individual", tools.initCycle, creator.Individual,                 (toolbox.attr_n_estimators, toolbox.attr_max_depth,                  toolbox.attr_min_samples_split, toolbox.attr_min_samples_leaf,                  toolbox.attr_max_features, toolbox.attr_bootstrap, toolbox.attr_threshold), n=1)toolbox.register("population", tools.initRepeat, list, toolbox.individual)print("✅ Algoritmo Genético configurado!")print(f"   - Genes: 7 (6 hiperparâmetros + 1 threshold)")print(f"   - Fitness: Maximizar Recall")

## 🧬 FASE 2: Otimização com Algoritmos GenéticosAgora vamos otimizar o modelo **Random Forest** usando **Algoritmos Genéticos (AG)** com a biblioteca **DEAP**.### 🎯 Objetivos da Otimização:1. **Maximizar o Recall** (função fitness)2. **Otimizar hiperparâmetros** do Random Forest3. **Otimizar o limiar de classificação** (threshold)### 🧬 Configuração do AG:- **População**: 50 indivíduos- **Gerações**: 30- **Crossover**: Two-Point (70%)- **Mutação**: Customizada com sigma específico (20%)- **Seleção**: Torneio (tournsize=3)- **Validação**: 3-fold Stratified Cross-Validation

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))axes = axes.ravel()for idx, (nome, resultado) in enumerate(resultados_baseline.items()):    cm = confusion_matrix(y_teste, resultado['y_pred'])    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], cbar=False)    axes[idx].set_title(f'{nome}\nRecall: {resultado["Recall"]:.2f} | F1: {resultado["F1-Score"]:.2f}',                         fontsize=12, fontweight='bold')    axes[idx].set_xlabel('Predito')    axes[idx].set_ylabel('Real')plt.suptitle('Matrizes de Confusão - Modelos Baseline', fontsize=16, fontweight='bold', y=1.00)plt.tight_layout()plt.show()

In [ ]:
resultados_baseline = {}limiar_baseline = 0.3print("📊 Avaliação dos Modelos Baseline (Limiar = 0.3)\n")print("="*80)for nome, modelo in modelos_baseline.items():    if hasattr(modelo, "predict_proba"):        y_proba = modelo.predict_proba(X_teste)[:, 1]        y_pred = (y_proba >= limiar_baseline).astype(int)    else:        y_pred = modelo.predict(X_teste)        recall = recall_score(y_teste, y_pred)    f1 = f1_score(y_teste, y_pred)    acc = accuracy_score(y_teste, y_pred)        resultados_baseline[nome] = {        'Recall': recall,        'F1-Score': f1,        'Acurácia': acc,        'y_pred': y_pred    }        print(f"\n{nome}:")    print(f"   Recall (Sensibilidade): {recall:.2f}")    print(f"   F1-Score: {f1:.2f}")    print(f"   Acurácia: {acc:.2f}")print("\n" + "="*80)df_resultados_baseline = pd.DataFrame(resultados_baseline).T[['Recall', 'F1-Score', 'Acurácia']]print("\n📊 Resumo dos Resultados Baseline:\n")print(df_resultados_baseline.sort_values('Recall', ascending=False))melhor_modelo_baseline = df_resultados_baseline['Recall'].idxmax()print(f"\n🏆 Melhor Modelo Baseline: {melhor_modelo_baseline}")print(f"   Recall: {df_resultados_baseline.loc[melhor_modelo_baseline, 'Recall']:.2f}")

In [ ]:
from sklearn.linear_model import LogisticRegressionfrom sklearn.tree import DecisionTreeClassifierfrom sklearn.ensemble import RandomForestClassifierimport lightgbm as lgbmodelos_baseline = {    'Regressão Logística': LogisticRegression(class_weight='balanced', random_state=42),    'Árvore de Decisão': DecisionTreeClassifier(class_weight='balanced', random_state=42),    'Random Forest': RandomForestClassifier(class_weight='balanced', n_estimators=100, random_state=42),    'LightGBM': lgb.LGBMClassifier(class_weight='balanced', random_state=42, verbose=-1)}print("🤖 Treinando Modelos Baseline...\n")for nome, modelo in modelos_baseline.items():    modelo.fit(X_treino_res, y_treino_res)    print(f"   ✅ {nome} treinado")print("\n✅ Todos os modelos baseline foram treinados com sucesso!")

## 📈 FASE 1: Modelos BaselineVamos treinar e avaliar 4 modelos de classificação baseline:1. **Regressão Logística**2. **Árvore de Decisão**3. **Random Forest**4. **LightGBM****Métrica Principal**: Recall (Sensibilidade) - capacidade de detectar casos positivos de diabetes

In [ ]:
X_treino, X_teste, y_treino, y_teste = train_test_split(    X_escalado, y, test_size=0.2, random_state=42, stratify=y)print("✂️ Divisão dos Dados:")print(f"   - Treino: {X_treino.shape[0]} amostras ({X_treino.shape[0]/len(y):.1%})")print(f"   - Teste: {X_teste.shape[0]} amostras ({X_teste.shape[0]/len(y):.1%})")print(f"\n📊 Distribuição ANTES do SMOTE (Treino):")unique, counts = np.unique(y_treino, return_counts=True)for label, count in zip(unique, counts):    print(f"   - Classe {label}: {count} ({count/len(y_treino):.1%})")smote = SMOTE(random_state=42)X_treino_res, y_treino_res = smote.fit_resample(X_treino, y_treino)print(f"\n📊 Distribuição APÓS o SMOTE (Treino):")unique, counts = np.unique(y_treino_res, return_counts=True)for label, count in zip(unique, counts):    print(f"   - Classe {label}: {count} ({count/len(y_treino_res):.1%})")print(f"\n✅ SMOTE aplicado com sucesso!")print(f"   - Treino balanceado: {X_treino_res.shape}")print(f"   - Teste (não modificado): {X_teste.shape}")

## ✂️ Etapa 3: Divisão dos Dados e Balanceamento (SMOTE)Para lidar com o desbalanceamento da classe alvo, vamos:1. Dividir os dados em treino (80%) e teste (20%)2. Aplicar SMOTE (Synthetic Minority Over-sampling Technique) apenas no conjunto de treino

In [ ]:
X = df_processado.drop('Outcome', axis=1)y = df_processado['Outcome'].valuesnomes_features = X.columns.tolist()print("📊 Separação de Features e Target:")print(f"   - X (Features): {X.shape}")print(f"   - y (Target): {y.shape}")print(f"\n📝 Features utilizadas ({len(nomes_features)}):")for i, feature in enumerate(nomes_features, 1):    print(f"   {i}. {feature}")scaler = RobustScaler()X_escalado = scaler.fit_transform(X)print(f"\n✅ Escalonamento aplicado com RobustScaler")print(f"   - Robusto a outliers (usa mediana e IQR)")print(f"   - Shape final: {X_escalado.shape}")

In [ ]:
df_processado = df.copy()print("🗑️ Removendo features com muitos valores ausentes...")df_processado = df_processado.drop(['SkinThickness', 'Insulin'], axis=1)print(f"   ✅ Features removidas: SkinThickness, Insulin")print(f"   📊 Dimensões após remoção: {df_processado.shape}")print("\n🔧 Tratando valores zero biologicamente impossíveis...")colunas_tratar = ['Glucose', 'BloodPressure', 'BMI']for col in colunas_tratar:    zeros_antes = (df_processado[col] == 0).sum()    df_processado[col] = df_processado[col].replace(0, np.nan)    print(f"   - {col}: {zeros_antes} zeros convertidos para NaN")imputador = SimpleImputer(strategy='median')df_processado[colunas_tratar] = imputador.fit_transform(df_processado[colunas_tratar])print(f"   ✅ Valores NaN imputados com a mediana")print("\n🧬 Criando features de engenharia...")df_processado['idade_maior_45'] = (df_processado['Age'] >= 45).astype(int)df_processado['imc_obeso'] = (df_processado['BMI'] >= 30).astype(int)df_processado['idade_bmi'] = df_processado['Age'] * df_processado['BMI']df_processado['glucose_bmi'] = df_processado['Glucose'] * df_processado['BMI']print(f"   ✅ Features criadas:")print(f"      - idade_maior_45 (indicador de risco)")print(f"      - imc_obeso (indicador de obesidade)")print(f"      - idade_bmi (interação Idade x IMC)")print(f"      - glucose_bmi (interação Glicose x IMC)")print(f"\n📊 Dimensões finais: {df_processado.shape}")df_processado.head()

## 🧹 Etapa 2: Pré-processamento dos DadosNesta etapa, vamos:1. Remover features com muitos valores ausentes (SkinThickness, Insulin)2. Tratar valores zero biologicamente impossíveis3. Criar features de engenharia (interações e indicadores de risco)4. Aplicar escalonamento robusto

In [ ]:
plt.figure(figsize=(12, 8))sns.heatmap(df.corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f', linewidths=0.5)plt.title('Matriz de Correlação entre Features', fontsize=16, fontweight='bold')plt.tight_layout()plt.show()print("🔍 Correlações Importantes:")print("   - Glucose tem a maior correlação com Outcome (0.47)")print("   - Age e Pregnancies têm correlação moderada (0.54)")print("   - BMI e SkinThickness têm correlação alta (0.65)")

In [ ]:
df.hist(bins=20, figsize=(15, 10), edgecolor='black')plt.suptitle('Distribuição das Features', fontsize=16, fontweight='bold', y=1.00)plt.tight_layout()plt.show()print("🔍 Observações:")print("   - Glucose, BloodPressure, BMI têm valores zero (biologicamente impossíveis)")print("   - Age tem distribuição assimétrica (mais jovens)")print("   - Pregnancies tem muitos zeros (esperado para homens e mulheres jovens)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))sns.countplot(x='Outcome', data=df, ax=axes[0], palette='viridis')axes[0].set_title('Distribuição da Variável Alvo (Outcome)', fontsize=14, fontweight='bold')axes[0].set_xlabel('Outcome (0=Não Diabético, 1=Diabético)')axes[0].set_ylabel('Contagem')for container in axes[0].containers:    axes[0].bar_label(container)df['Outcome'].value_counts().plot(kind='pie', autopct='%1.1f%%', ax=axes[1], colors=['#2ecc71', '#e74c3c'])axes[1].set_title('Proporção de Diabéticos vs Não Diabéticos', fontsize=14, fontweight='bold')axes[1].set_ylabel('')plt.tight_layout()plt.show()print("⚠️ Observação: Dataset desbalanceado - 65.1% Não Diabéticos vs 34.9% Diabéticos")

In [ ]:
print("📊 Informações do Dataset:\n")print(df.info())print("\n" + "="*80)print("\n📈 Estatísticas Descritivas:\n")print(df.describe())print("\n" + "="*80)print("\n🔍 Valores Ausentes:\n")print(df.isnull().sum())print("\n" + "="*80)print("\n⚖️ Distribuição da Variável Alvo (Outcome):\n")print(df['Outcome'].value_counts())print(f"\nProporção de Diabéticos: {df['Outcome'].mean():.2%}")

### 🔍 Análise Exploratória Inicial

In [ ]:
from pathlib import Pathlocal_path = Path('pipeline/dados/diabetes.csv')try:    df = pd.read_csv(local_path)    print(f"✅ Dataset carregado localmente de: {local_path}")except:    print("⚠️ Arquivo local não encontrado. Baixando do Kaggle...")    import kagglehub    endereco_de_origem = kagglehub.dataset_download(handle='uciml/pima-indians-diabetes', force_download=True)    diretorio_de_origem = Path(endereco_de_origem).resolve()    lista_dados_csv = []    for item in diretorio_de_origem.iterdir():        if item.is_file() and item.suffix.lower() == '.csv':            lista_dados_csv.append(pd.read_csv(item))    df = pd.concat(lista_dados_csv, axis=0, ignore_index=True)    print(f"✅ Dataset baixado do Kaggle")print(f"\n📊 Dimensões do Dataset: {df.shape}")print(f"   - {df.shape[0]} pacientes")print(f"   - {df.shape[1]} features (incluindo target)")df.head()

## 📥 Etapa 1: Carregamento dos DadosVamos carregar o dataset do Kaggle e realizar uma análise exploratória inicial.

In [ ]:
import warningswarnings.filterwarnings('ignore')import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import RobustScalerfrom sklearn.impute import SimpleImputerfrom sklearn.metrics import recall_score, f1_score, accuracy_score, confusion_matrix, classification_reportfrom imblearn.over_sampling import SMOTEimport shapsns.set_style('whitegrid')plt.rcParams['figure.figsize'] = (12, 6)print("✅ Bibliotecas importadas com sucesso!")

# 🧬 Tech Challenge Fase 2: Diagnóstico de Diabetes com Algoritmos Genéticos e LLMs## 📋 Visão Geral do ProjetoEste notebook apresenta a **Fase 2** do projeto de diagnóstico de diabetes, que evolui do modelo baseline (Fase 1) para um sistema otimizado utilizando:1. **Algoritmos Genéticos (AG)** para otimização de hiperparâmetros e limiar de classificação2. **Large Language Models (LLMs)** para interpretabilidade e geração de insights médicos acionáveis3. **SHAP (SHapley Additive exPlanations)** para explicabilidade dos modelos### 🎯 Objetivos- **Maximizar o Recall (Sensibilidade)**: Detectar o máximo de casos positivos de diabetes- **Otimizar via Algoritmos Genéticos**: Encontrar os melhores hiperparâmetros e limiar de classificação- **Gerar Insights Acionáveis**: Traduzir resultados técnicos em recomendações médicas práticas### 📊 Dataset- **Fonte**: Pima Indians Diabetes Dataset (Kaggle)- **Características**: 768 pacientes, 8 features clínicas- **Variável Alvo**: Outcome (0 = Não Diabético, 1 = Diabético)---